In [23]:
import pandas as pd
import numpy as np



In [24]:
df = pd.read_csv("Spam_SMS.csv")

In [25]:
df.head()

,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [26]:
df.isnull().sum()

,0
Class,0
Message,0


In [27]:
df.duplicated().sum()

np.int64(415)

In [28]:
df.drop_duplicates(inplace=True , keep='first')

In [29]:
df.info

<bound method DataFrame.info of      Class                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5569  spam  This is the 2nd time we have tried 2 contact u...
5570   ham               Will ü b going to esplanade fr home?
5571   ham  Pity, * was in mood for that. So...any other s...
5572   ham  The guy did some bitching but I acted like i'd...
5573   ham                         Rofl. Its true to its name

[5159 rows x 2 columns]>

In [30]:
unique = df["Class"].unique()
numbers ={}
i = 0
for target in unique:
    numbers[target] = i
    i += 1


In [31]:
df['Class'] = df['Class'].map(numbers)

In [32]:
df.head()

,Class,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [33]:
df["Class"].value_counts()

,count
Class,
0,4518
1,641


In [34]:
df['Message']=df['Message'].apply(lambda x : x.lower())


In [35]:
import string

def remove_pun(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [36]:
df['Message']=df['Message'].apply(remove_pun)


In [37]:
def remove_num(txt):
    new =''
    for i  in txt:
        if not i.isdigit():
            new =new + i
    return new
df['Message'] = df["Message"].apply(remove_num)



In [38]:
import nltk
from nltk.corpus import stopwords
from  nltk.tokenize import word_tokenize

In [46]:


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [47]:
stopwords = set(stopwords.words('english'))

AttributeError: 'set' object has no attribute 'words'

In [48]:
df.head()

,Class,Message
0,0,go until jurong point crazy available only in ...
1,0,ok lar joking wif u oni
2,1,free entry in a wkly comp to win fa cup final...
3,0,u dun say so early hor u c already then say
4,0,nah i dont think he goes to usf he lives aroun...


In [42]:
len(stopwords)

198

In [43]:
def remove_stopw(txt):
    if txt is None:
        return ''

    words = word_tokenize(txt)

    cleaned_txt = []

    for i in words:
        if i not in stopwords:
            cleaned_txt.append(i)

    return ' '.join(cleaned_txt)

In [45]:
df['Message']= df['Message'].apply(remove_stopw)

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
df.head()

,Class,Message
0,0,go jurong point crazy available bugis n great ...
1,0,ok lar joking wif u oni
2,1,free entry wkly comp win fa cup final tkts st ...
3,0,u dun say early hor u c already say
4,0,nah dont think goes usf lives around though


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer , CountVectorizer
vector_tf= TfidfVectorizer()
vector_bow = CountVectorizer()


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split( df['Message'], df['Class'] , test_size=0.2 ,random_state=42 )

In [ ]:
X_train_bow = vector_bow.fit_transform(X_train)
X_test_bow = vector_bow.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,f1_score,classification_report

In [ ]:
models=[
    ('Lgr' , LogisticRegression(class_weight='balanced')),
    ('Mnb' , MultinomialNB(alpha=1.0)),
    ('Svm', SVC(kernel='rbf'))
]

In [ ]:
result =[]
for name , model in models:
    model.fit(X_train_bow ,y_train)
    y_pred_bow =model.predict(X_test_bow)
    Accuracy = accuracy_score(y_test, y_pred_bow)
    F1 = f1_score(y_test, y_pred_bow)
    result.append({
        'model': name,
        'Accuracy': round(Accuracy, 4),
        'F1': round(F1, 4)
    })

In [ ]:
result

[{'model': 'Lgr', 'Accuracy': 0.9738, 'F1': 0.8861},
 {'model': 'Mnb', 'Accuracy': 0.9777, 'F1': 0.9061},
 {'model': 'Svm', 'Accuracy': 0.9748, 'F1': 0.8829}]

In [ ]:
X_train_tf = vector_tf.fit_transform(X_train)
X_test_tf = vector_tf.transform(X_test)

In [ ]:
result_tf =[]
for name , model in models:
    model.fit(X_train_tf ,y_train)
    y_pred_tf =model.predict(X_test_tf)
    Accuracy = accuracy_score(y_test, y_pred_tf)
    F1 = f1_score(y_test, y_pred_tf)
    result_tf.append({
        'model': name,
        'Accuracy': round(Accuracy, 4),
        'F1': round(F1, 4)
    })

In [ ]:
result_tf

[{'model': 'Lgr', 'Accuracy': 0.9709, 'F1': 0.881},
 {'model': 'Mnb', 'Accuracy': 0.968, 'F1': 0.8436},
 {'model': 'Svm', 'Accuracy': 0.9738, 'F1': 0.8767}]

In [ ]:
import joblib


model_mnb = MultinomialNB(alpha=1.0)
model_mnb.fit(X_train_bow, y_train)

joblib.dump(model_mnb, 'spam_mnb_model.pkl')
joblib.dump(vector_bow, 'bow_vectorizer.pkl')

# later, to load:
model_mnb = joblib.load('spam_mnb_model.pkl')
vector_bow = joblib.load('bow_vectorizer.pkl')